In [1]:
import pandas as pd

file = r"E:\NUS\galaxy\rc3\asu2.tsv"

# 读入，分号分隔，跳过注释行
rc3 = pd.read_csv(file, sep=";", comment="#")

# 强制把 RA/DEC 转换成 float，不可转换的变 NaN
rc3["_RAJ2000"] = pd.to_numeric(rc3["_RAJ2000"], errors="coerce")
rc3["_DEJ2000"] = pd.to_numeric(rc3["_DEJ2000"], errors="coerce")

# 丢掉无效行
rc3 = rc3.dropna(subset=["_RAJ2000", "_DEJ2000"]).reset_index(drop=True)

print(rc3.head())


   _RAJ2000   _DEJ2000      RA2000     DE2000          altname          PGC  \
0  0.007852  47.274382  00 00 01.8  +47 16 28   UGC 12889       PGC    2      
1  0.036593  -6.374703  00 00 08.7  -06 22 29   MCG -1- 1- 16   PGC   12      
2  0.089093  -2.610543  00 00 21.3  -02 36 38   MCG -1- 1- 20   PGC   23      
3  0.094195 -80.791574  00 00 22.5  -80 47 30   ESO   12- 12    PGC   30      
4  0.100348  39.499948  00 00 24.0  +39 30 00   UGC 12894       PGC   35      

      type     T    D25    R25     BT BT_code     cz  
0  .SBT3..   3.0   1.33   0.07                   NaN  
1  .S..1P?   1.0   1.18   0.74                  6493  
2  .E+..*.  -4.0   1.15   0.25                   NaN  
3  PSXT4P.   4.0   1.19   0.07                  7900  
4  .I..9..  10.0   0.96   0.00                   NaN  


In [2]:
# === Cell 1: imports & paths ===
import os, re, math, numpy as np, pandas as pd, requests
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

# 你的路径
rc3_tsv   = r"E:\NUS\galaxy\rc3\asu2.tsv"             # RC3 TSV (VizieR 导出)
outdir    = r"E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4" # 新的输出目录
report_csv = r"E:\NUS\galaxy\rc3\centered_check_report3.csv"
os.makedirs(outdir, exist_ok=True)

# HiPS2FITS 服务与 SDSS DR9 r 波段 HiPS
HIPS2FITS = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits"
HIPS_SDSS_R = "CDS/P/SDSS9/r"   # 也可换成 "CDS/P/SDSS9/g", "CDS/P/SDSS9/i"


In [3]:
# === Cell 1: imports & paths ===
import os, re, math, numpy as np, pandas as pd, requests
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

# 你的路径
rc3_tsv    = r"E:\NUS\galaxy\rc3\asu2.tsv"                 # RC3 TSV (VizieR 导出)
outdir     = r"E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4"  # 总输出目录
report_csv = r"E:\NUS\galaxy\rc3\centered_check_report4.csv"
os.makedirs(outdir, exist_ok=True)

# 选择 SDSS DR9 的五个波段（SDSS 无 b 波段）
BANDS = ['u', 'g', 'r', 'i', 'z']

# HiPS2FITS 服务与 SDSS DR9 HiPS 集合（不同 band 仅最后一段不同）
HIPS2FITS = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits"
HIPS_SDSS = {b: f"CDS/P/SDSS9/{b}" for b in BANDS}

# 为每个波段创建单独输出子目录（可选）
band_outdirs = {b: os.path.join(outdir, b) for b in BANDS}
for d in band_outdirs.values():
    os.makedirs(d, exist_ok=True)

# 小工具：根据目标名与波段生成输出文件名
def make_outpath(base_name: str, band: str) -> str:
    """
    base_name: 例如 'UGC_12889_0'（你后续代码里生成的目标名）
    band:      'u'/'g'/'r'/'i'/'z'
    return:    该 band 的 FITS 完整保存路径
    """
    return os.path.join(band_outdirs[band], f"{base_name}_{band}.fits")


In [4]:
# === Cell 2: load & clean RC3 TSV ===
rc3 = pd.read_csv(rc3_tsv, sep=";", comment="#")

# 强制转数值；无效坐标丢弃
rc3["_RAJ2000"] = pd.to_numeric(rc3["_RAJ2000"], errors="coerce")
rc3["_DEJ2000"] = pd.to_numeric(rc3["_DEJ2000"], errors="coerce")
rc3 = rc3.dropna(subset=["_RAJ2000", "_DEJ2000"]).reset_index(drop=True)

print(f"[INFO] RC3 rows: {len(rc3)}")
print(rc3.head())


[INFO] RC3 rows: 9999
   _RAJ2000   _DEJ2000      RA2000     DE2000          altname          PGC  \
0  0.007852  47.274382  00 00 01.8  +47 16 28   UGC 12889       PGC    2      
1  0.036593  -6.374703  00 00 08.7  -06 22 29   MCG -1- 1- 16   PGC   12      
2  0.089093  -2.610543  00 00 21.3  -02 36 38   MCG -1- 1- 20   PGC   23      
3  0.094195 -80.791574  00 00 22.5  -80 47 30   ESO   12- 12    PGC   30      
4  0.100348  39.499948  00 00 24.0  +39 30 00   UGC 12894       PGC   35      

      type     T    D25    R25     BT BT_code     cz  
0  .SBT3..   3.0   1.33   0.07                   NaN  
1  .S..1P?   1.0   1.18   0.74                  6493  
2  .E+..*.  -4.0   1.15   0.25                   NaN  
3  PSXT4P.   4.0   1.19   0.07                  7900  
4  .I..9..  10.0   0.96   0.00                   NaN  


In [5]:
# === Cell 3: helpers ===
def sanitize_name(name: str, fallback: str):
    if not isinstance(name, str) or not name.strip():
        return fallback
    s = name.strip().replace(" ", "_")
    # 仅保留安全字符
    s = re.sub(r"[^A-Za-z0-9_\-+]", "_", s)
    return s if s else fallback

def d25_arcmin(row):
    if "D25" in row and pd.notna(row["D25"]):
        try:
            return 0.1 * (10.0 ** float(row["D25"]))  # RC3 定义
        except Exception:
            return np.nan
    return np.nan

def choose_fov_deg(row, min_arcmin=3.0, max_arcmin=30.0, scale_factor=4.0):
    """
    FoV（度）= max(min_arcmin, min(max_arcmin, scale_factor * A25)) / 60
    让星系直径约占 FoV 的 ~1/scale_factor
    """
    a25 = d25_arcmin(row)
    if np.isnan(a25):
        fov_arcmin = 5.0  # 无 D25 时默认 5'
    else:
        fov_arcmin = max(min_arcmin, min(max_arcmin, scale_factor * a25))
    return fov_arcmin / 60.0  # 转度


In [6]:
# === Cell 4: download via HiPS2FITS (SDSS r) ===
width, height = 64, 64   # 输出像素尺寸；像素尺度由 FoV/width 决定
session = requests.Session()
session.headers.update({"User-Agent": "rc3-sdss-cutouts/1.0"})

# 先少量测试；确认 OK 后，可去掉 .head(N) 全量下载
subset = rc3.head(150)

for i, row in subset.iterrows():
    ra, dec = float(row["_RAJ2000"]), float(row["_DEJ2000"])
    altname = sanitize_name(row.get("altname", ""), f"RC3_{i}")
    fov_deg = choose_fov_deg(row)  # FoV（度）

    fname = os.path.join(outdir, f"{altname}_{i}.fits")
    if os.path.exists(fname):
        print(f"[SKIP] {fname}")
        continue

    params = {
        "hips": HIPS_SDSS_R,
        "ra": ra, "dec": dec,
        "width": width, "height": height,
        "fov": fov_deg,
        "projection": "TAN",
        "format": "fits",
        "coordsys": "icrs",
    }

    try:
        r = session.get(HIPS2FITS, params=params, timeout=90)
        r.raise_for_status()
        with open(fname, "wb") as f:
            f.write(r.content)
        print(f"[OK] {fname}  FoV={fov_deg*60:.1f}'")
    except Exception as e:
        print(f"[FAIL] idx={i} {altname}  {e}")


[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12889_0.fits  FoV=8.6'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\MCG_-1-_1-_16_1.fits  FoV=6.1'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\MCG_-1-_1-_20_2.fits  FoV=5.7'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\ESO___12-_12_3.fits  FoV=6.2'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12894_4.fits  FoV=3.6'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12893_5.fits  FoV=6.8'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12892_6.fits  FoV=3.3'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\ESO__293-_27_7.fits  FoV=7.6'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12895_8.fits  FoV=3.2'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12897_9.fits  FoV=4.6'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12898_10.fits  FoV=3.6'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\MCG__5-_1-_21_11.fits  FoV=3.1'
[OK] E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\UGC_12899_12.fits  FoV=3.8'
[OK] E:\NUS\galax

KeyboardInterrupt: 

In [9]:
# === Cell 4: download via HiPS2FITS (SDSS u,g,r,i,z) ===
width, height = 64, 64   # 输出像素尺寸；像素尺度由 FoV/width 决定
session = requests.Session()
session.headers.update({"User-Agent": "rc3-sdss-cutouts/1.0"})

# 先少量测试；确认 OK 后，可去掉 .head(N) 全量下载
subset = rc3.head(150)

for i, row in subset.iterrows():
    ra, dec = float(row["_RAJ2000"]), float(row["_DEJ2000"])
    altname = sanitize_name(row.get("altname", ""), f"RC3_{i}")
    fov_deg = choose_fov_deg(row)  # FoV（度）
    base = f"{altname}_{i}"        # 统一的基础名，便于多波段区分

    for band in BANDS:  # e.g., ['u','g','r','i','z']
        hips_id = HIPS_SDSS[band]
        fname = make_outpath(base, band)  # 如 .../r/UGC_12889_0_r.fits

        if os.path.exists(fname):
            print(f"[SKIP] {band} {fname}")
            continue

        params = {
            "hips": hips_id,
            "ra": ra, "dec": dec,
            "width": width, "height": height,
            "fov": fov_deg,
            "projection": "TAN",
            "format": "fits",
            "coordsys": "icrs",
        }

        try:
            r = session.get(HIPS2FITS, params=params, timeout=90)
            r.raise_for_status()
            with open(fname, "wb") as f:
                f.write(r.content)
            print(f"[OK] {band} {fname}  FoV={fov_deg*60:.1f}'")
        except Exception as e:
            print(f"[FAIL] {band} idx={i} {altname}  {e}")


[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC_12889_0_u.fits  FoV=8.6'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\UGC_12889_0_g.fits  FoV=8.6'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC_12889_0_r.fits  FoV=8.6'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC_12889_0_i.fits  FoV=8.6'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC_12889_0_z.fits  FoV=8.6'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG_-1-_1-_16_1_u.fits  FoV=6.1'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG_-1-_1-_16_1_g.fits  FoV=6.1'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\MCG_-1-_1-_16_1_r.fits  FoV=6.1'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\MCG_-1-_1-_16_1_i.fits  FoV=6.1'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\MCG_-1-_1-_16_1_z.fits  FoV=6.1'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG_-1-_1-_20_2_u.fits  FoV=5.7'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG_-1-_1-_20_2_g.fits  FoV=5.7'
[OK] r E:\NU

[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG__1-_1-__9_20_u.fits  FoV=3.9'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG__1-_1-__9_20_g.fits  FoV=3.9'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\MCG__1-_1-__9_20_r.fits  FoV=3.9'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\MCG__1-_1-__9_20_i.fits  FoV=3.9'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\MCG__1-_1-__9_20_z.fits  FoV=3.9'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC_12902_21_u.fits  FoV=4.5'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\UGC_12902_21_g.fits  FoV=4.5'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC_12902_21_r.fits  FoV=4.5'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC_12902_21_i.fits  FoV=4.5'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC_12902_21_z.fits  FoV=4.5'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG__2-_1-__9_22_u.fits  FoV=3.5'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG__2-_1-__9_22_g.fits  FoV=3.5'


[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\ESO__349-_19_40_u.fits  FoV=4.6'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\ESO__349-_19_40_g.fits  FoV=4.6'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\ESO__349-_19_40_r.fits  FoV=4.6'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\ESO__349-_19_40_i.fits  FoV=4.6'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\ESO__349-_19_40_z.fits  FoV=4.6'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\ESO__409-__3_41_u.fits  FoV=5.4'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\ESO__409-__3_41_g.fits  FoV=5.4'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\ESO__409-__3_41_r.fits  FoV=5.4'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\ESO__409-__3_41_i.fits  FoV=5.4'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\ESO__409-__3_41_z.fits  FoV=5.4'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG_-3-_1-_15_42_u.fits  FoV=30.0'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG_-3-_1-_15_42_g.fits

[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\ESO__349-_21_59_z.fits  FoV=4.4'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC_____4_60_u.fits  FoV=4.1'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\UGC_____4_60_g.fits  FoV=4.1'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC_____4_60_r.fits  FoV=4.1'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC_____4_60_i.fits  FoV=4.1'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC_____4_60_z.fits  FoV=4.1'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\ESO__293-_31_61_u.fits  FoV=4.2'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\ESO__293-_31_61_g.fits  FoV=4.2'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\ESO__293-_31_61_r.fits  FoV=4.2'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\ESO__293-_31_61_i.fits  FoV=4.2'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\ESO__293-_31_61_z.fits  FoV=4.2'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC_____5_62_u.fits  FoV=7.1'
[OK] g E:\

[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC____17_79_z.fits  FoV=9.8'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC____15_80_u.fits  FoV=5.0'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\UGC____15_80_g.fits  FoV=5.0'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC____15_80_r.fits  FoV=5.0'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC____15_80_i.fits  FoV=5.0'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC____15_80_z.fits  FoV=5.0'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC____16_81_u.fits  FoV=6.8'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\UGC____16_81_g.fits  FoV=6.8'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC____16_81_r.fits  FoV=6.8'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC____16_81_i.fits  FoV=6.8'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC____16_81_z.fits  FoV=6.8'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC____19_82_u.fits  FoV=14.2'
[OK] g E:\NUS\galaxy\rc3\sd

[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\MCG_-1-_1-_30_99_z.fits  FoV=5.7'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\ESO__409-_12_100_u.fits  FoV=4.9'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\ESO__409-_12_100_g.fits  FoV=4.9'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\ESO__409-_12_100_r.fits  FoV=4.9'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\ESO__409-_12_100_i.fits  FoV=4.9'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\ESO__409-_12_100_z.fits  FoV=4.9'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\CGCG_477-_49_101_u.fits  FoV=5.0'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\CGCG_477-_49_101_g.fits  FoV=5.0'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\CGCG_477-_49_101_r.fits  FoV=5.0'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\CGCG_477-_49_101_i.fits  FoV=5.0'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\CGCG_477-_49_101_z.fits  FoV=5.0'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG_-3-_1-_18_

[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC____38_119_r.fits  FoV=4.8'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC____38_119_i.fits  FoV=4.8'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC____38_119_z.fits  FoV=4.8'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG__5-_1-_31_120_u.fits  FoV=3.1'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG__5-_1-_31_120_g.fits  FoV=3.1'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\MCG__5-_1-_31_120_r.fits  FoV=3.1'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\MCG__5-_1-_31_120_i.fits  FoV=3.1'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\MCG__5-_1-_31_120_z.fits  FoV=3.1'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\ESO__193-_19_121_u.fits  FoV=7.8'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\ESO__193-_19_121_g.fits  FoV=7.8'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\ESO__193-_19_121_r.fits  FoV=7.8'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\ESO__193-_19_121_i

[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\ESO__293-_34_139_g.fits  FoV=12.6'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\ESO__293-_34_139_r.fits  FoV=12.6'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\ESO__293-_34_139_i.fits  FoV=12.6'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\ESO__293-_34_139_z.fits  FoV=12.6'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\UGC____46_140_u.fits  FoV=3.0'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\UGC____46_140_g.fits  FoV=3.0'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\UGC____46_140_r.fits  FoV=3.0'
[OK] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\i\UGC____46_140_i.fits  FoV=3.0'
[OK] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\z\UGC____46_140_z.fits  FoV=3.0'
[OK] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\u\MCG_-7-_1-_10_141_u.fits  FoV=3.0'
[OK] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\g\MCG_-7-_1-_10_141_g.fits  FoV=3.0'
[OK] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\r\MCG_-7-_1-_10_141_r.fit

In [7]:
# === Cell 4: download via HiPS2FITS (SDSS u,g,r,i,z) ===
width, height = 64, 64   # 输出像素尺寸；像素尺度由 FoV/width 决定
session = requests.Session()
session.headers.update({"User-Agent": "rc3-sdss-cutouts/1.0"})

# 取第160到第165个（含端点）
start_n, end_n = 160, 175
subset = rc3.reset_index(drop=True).iloc[start_n-1:end_n]
print(f"[INFO] 将下载第 {start_n}–{end_n} 个，共 {len(subset)} 条")

for i, row in subset.iterrows():
    ra, dec = float(row["_RAJ2000"]), float(row["_DEJ2000"])
    altname = sanitize_name(row.get("altname", ""), f"RC3_{i}")
    fov_deg = choose_fov_deg(row)  # FoV（度）
    base = f"{altname}_{i}"        # 统一的基础名，便于多波段区分

    for band in BANDS:  # e.g., ['u','g','r','i','z']
        hips_id = HIPS_SDSS[band]
        fname = make_outpath(base, band)  # 如 .../r/UGC_12889_0_r.fits

        if os.path.exists(fname):
            print(f"[SKIP] {band} {fname}")
            continue

        params = {
            "hips": hips_id,
            "ra": ra, "dec": dec,
            "width": width, "height": height,
            "fov": fov_deg,
            "projection": "TAN",
            "format": "fits",
            "coordsys": "icrs",
        }

        try:
            r = session.get(HIPS2FITS, params=params, timeout=90)
            r.raise_for_status()
            with open(fname, "wb") as f:
                f.write(r.content)
            print(f"[OK] {band} {fname}  FoV={fov_deg*60:.1f}'")
        except Exception as e:
            print(f"[FAIL] {band} idx={i} {altname}  {e}")


[INFO] 将下载第 160–175 个，共 16 条
[SKIP] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\u\RC3_159_159_u.fits
[SKIP] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\g\RC3_159_159_g.fits
[SKIP] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\r\RC3_159_159_r.fits
[SKIP] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\i\RC3_159_159_i.fits
[SKIP] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\z\RC3_159_159_z.fits
[SKIP] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\u\UGC____56_160_u.fits
[SKIP] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\g\UGC____56_160_g.fits
[SKIP] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\r\UGC____56_160_r.fits
[SKIP] i E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\i\UGC____56_160_i.fits
[SKIP] z E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\z\UGC____56_160_z.fits
[SKIP] u E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\u\ESO__293-_37_161_u.fits
[SKIP] g E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\g\ESO__293-_37_161_g.fits
[SKIP] r E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits4\r\ESO__293-_37_161_r.fits
[SKIP] i E

In [23]:
# === Cell 5: centering check ===
fits_files = sorted([os.path.join(outdir, f) for f in os.listdir(outdir) if f.lower().endswith(".fits")])
rows = []
tol_5as, tol_30as = 5.0, 30.0

def parse_index_from_filename(name: str):
    m = re.search(r"_([0-9]+)\.fits$", name)
    return int(m.group(1)) if m else None

for path in fits_files:
    fname = os.path.basename(path)
    idx = parse_index_from_filename(fname)
    if idx is None or idx < 0 or idx >= len(rc3):
        rows.append(dict(file=fname, status="SKIP", note="no_valid_index"))
        continue

    ra = float(rc3.loc[idx, "_RAJ2000"])
    dec = float(rc3.loc[idx, "_DEJ2000"])
    altname = str(rc3.loc[idx, "altname"]).strip()

    try:
        with fits.open(path) as hdul:
            # 找 2D 图像平面
            hdu = next((h for h in hdul if h.data is not None and h.header.get("NAXIS",0) >= 2), None)
            if hdu is None:
                raise RuntimeError("No image HDU found")

            data = hdu.data
            w = WCS(hdu.header)

            H, W = int(data.shape[-2]), int(data.shape[-1])
            cx, cy = (W-1)/2.0, (H-1)/2.0

            # world -> pixel
            px, py = w.world_to_pixel_values(ra, dec)
            dx_pix, dy_pix = float(px - cx), float(py - cy)
            dr_pix = math.hypot(dx_pix, dy_pix)

            # 像素尺度（角秒/像素）
            sx_as, sy_as = (proj_plane_pixel_scales(w) * 3600.0)
            dx_as, dy_as = dx_pix * sx_as, dy_pix * sy_as
            dr_as = math.hypot(dx_as, dy_as)

            # 简单的“无覆盖/全零”判断
            invalid = False
            try:
                finite_ratio = np.isfinite(data).mean()
                nonzero_ratio = np.any(data != 0)
                if finite_ratio < 0.5 or not nonzero_ratio:
                    invalid = True
            except Exception:
                pass

            rows.append(dict(
                file=fname, idx=idx, altname=altname,
                ra=ra, dec=dec,
                dx_arcsec=dx_as, dy_arcsec=dy_as, dr_arcsec=dr_as,
                centered_le_5as = dr_as <= tol_5as,
                centered_le_30as = dr_as <= tol_30as,
                maybe_no_coverage = invalid,
                status="OK"
            ))

    except Exception as e:
        rows.append(dict(file=fname, idx=idx, altname=altname, ra=ra, dec=dec, status="ERROR", note=str(e)))

df = pd.DataFrame(rows)
df.to_csv(report_csv, index=False, float_format="%.3f", encoding="utf-8-sig")
print(f"[DONE] 写出报告: {report_csv}")

if "dr_arcsec" in df.columns:
    valid = df["dr_arcsec"].notna()
    rate5  = (df.loc[valid, "dr_arcsec"] <= tol_5as).mean() * 100
    rate30 = (df.loc[valid, "dr_arcsec"] <= tol_30as).mean() * 100
    print(f"Centered ≤5\": {rate5:.1f}%   |   ≤30\": {rate30:.1f}%")

    bad = df.loc[valid].sort_values("dr_arcsec", ascending=False).head(8)
    print("\n[Top off-center]")
    print(bad[["file","altname","dr_arcsec","dx_arcsec","dy_arcsec","maybe_no_coverage"]].to_string(index=False))


[DONE] 写出报告: E:\NUS\galaxy\rc3\centered_check_report2.csv


In [25]:
# === Cell 5: centering check (recursive + multi-band safe) ===
import os, re, math, numpy as np, pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

# 递归收集所有 .fits
fits_files = []
for root, _, files in os.walk(outdir):
    for f in files:
        if f.lower().endswith(".fits"):
            fits_files.append(os.path.join(root, f))
fits_files = sorted(fits_files)

rows = []
tol_5as, tol_30as = 5.0, 30.0

# 兼容两种命名：
# 1) 旧:  name_<idx>.fits
# 2) 新:  name_<idx>_<band>.fits  （例如 UGC_12889_0_r.fits）
pat_new = re.compile(r"_(\d+)_([ugriz])\.fits$", flags=re.I)
pat_old = re.compile(r"_(\d+)\.fits$", flags=re.I)

def parse_index_from_filename(name: str):
    m = pat_new.search(name)
    if m:
        return int(m.group(1))
    m = pat_old.search(name)
    if m:
        return int(m.group(1))
    return None

if len(fits_files) == 0:
    print(f"[WARN] 在 {outdir} 下未找到任何 FITS 文件。请确认 Cell 4 的输出路径与命名。")

for path in fits_files:
    fname = os.path.basename(path)
    idx = parse_index_from_filename(fname)
    if idx is None or idx < 0 or idx >= len(rc3):
        rows.append(dict(file=fname, status="SKIP", note="no_valid_index"))
        continue

    ra  = float(rc3.loc[idx, "_RAJ2000"])
    dec = float(rc3.loc[idx, "_DEJ2000"])
    altname = str(rc3.loc[idx, "altname"]).strip()

    try:
        with fits.open(path) as hdul:
            # 找 2D 图像平面
            hdu = next((h for h in hdul if (getattr(h, "data", None) is not None) and h.header.get("NAXIS", 0) >= 2), None)
            if hdu is None:
                raise RuntimeError("No image HDU found")

            data = np.asarray(hdu.data)
            H, W = int(data.shape[-2]), int(data.shape[-1])
            cx, cy = (W - 1) / 2.0, (H - 1) / 2.0

            # WCS（放宽并修复）
            w = WCS(hdu.header, relax=True)
            if not w.has_celestial:
                raise RuntimeError("WCS has no celestial component")

            # world -> pixel
            px, py = w.world_to_pixel_values(ra, dec)
            dx_pix, dy_pix = float(px - cx), float(py - cy)
            dr_pix = math.hypot(dx_pix, dy_pix)

            # 像素尺度（角秒/像素），必要时兜底
            try:
                sx_as, sy_as = (proj_plane_pixel_scales(w.celestial) * 3600.0)
            except Exception:
                # 兜底用 CDELT/CD
                cdelt1 = abs(hdu.header.get("CDELT1", np.nan))
                cdelt2 = abs(hdu.header.get("CDELT2", np.nan))
                if np.isfinite(cdelt1) and np.isfinite(cdelt2):
                    sx_as, sy_as = cdelt1 * 3600.0, cdelt2 * 3600.0
                else:
                    raise RuntimeError("Cannot determine pixel scale")

            dx_as, dy_as = dx_pix * sx_as, dy_pix * sy_as
            dr_as = math.hypot(dx_as, dy_as)

            # 简单覆盖性检查
            invalid = False
            try:
                finite_ratio = np.isfinite(data).mean()
                nonzero_any = bool(np.any((data != 0) & np.isfinite(data)))
                if finite_ratio < 0.5 or not nonzero_any:
                    invalid = True
            except Exception:
                pass

            rows.append(dict(
                file=fname, path=path, idx=idx, altname=altname,
                ra=ra, dec=dec,
                dx_arcsec=dx_as, dy_arcsec=dy_as, dr_arcsec=dr_as,
                centered_le_5as = dr_as <= tol_5as,
                centered_le_30as = dr_as <= tol_30as,
                maybe_no_coverage = invalid,
                status="OK"
            ))

    except Exception as e:
        rows.append(dict(file=fname, path=path, idx=idx, altname=altname, ra=ra, dec=dec,
                         status="ERROR", note=str(e)))

# 写出报告
df = pd.DataFrame(rows)
df.to_csv(report_csv, index=False, float_format="%.3f", encoding="utf-8-sig")
print(f"[DONE] 写出报告: {report_csv}  共 {len(df)} 行（含各波段文件）")

# 统计与预览
if "dr_arcsec" in df.columns and df["dr_arcsec"].notna().any():
    valid = df["dr_arcsec"].notna()
    rate5  = (df.loc[valid,  "dr_arcsec"] <= tol_5as ).mean() * 100
    rate30 = (df.loc[valid, "dr_arcsec"] <= tol_30as).mean() * 100
    print(f"Centered ≤5\": {rate5:.1f}%   |   ≤30\": {rate30:.1f}%")

    bad = df.loc[valid].sort_values("dr_arcsec", ascending=False).head(8)
    print("\n[Top off-center]")
    cols_show = [c for c in ["file","altname","dr_arcsec","dx_arcsec","dy_arcsec","maybe_no_coverage","status"] if c in bad.columns]
    print(bad[cols_show].to_string(index=False))
else:
    print("[WARN] 没有有效的 dr_arcsec 记录，可能没有解析到任何 FITS 或 WCS。")


[DONE] 写出报告: E:\NUS\galaxy\rc3\centered_check_report2.csv  共 750 行（含各波段文件）
Centered ≤5": 91.3%   |   ≤30": 100.0%

[Top off-center]
                   file       altname  dr_arcsec  dx_arcsec  dy_arcsec  maybe_no_coverage status
MCG_-3-_1-_15_42_i.fits MCG -3- 1- 15  19.887504 -14.062589 -14.062589               True     OK
MCG_-3-_1-_15_42_u.fits MCG -3- 1- 15  19.887504 -14.062589 -14.062589               True     OK
MCG_-3-_1-_15_42_r.fits MCG -3- 1- 15  19.887504 -14.062589 -14.062589               True     OK
MCG_-3-_1-_15_42_z.fits MCG -3- 1- 15  19.887504 -14.062589 -14.062589               True     OK
MCG_-3-_1-_15_42_g.fits MCG -3- 1- 15  19.887504 -14.062589 -14.062589               True     OK
    UGC_____8_69_z.fits     UGC     8  14.571953 -10.303926 -10.303926              False     OK
    UGC_____8_69_g.fits     UGC     8  14.571953 -10.303926 -10.303926              False     OK
    UGC_____8_69_u.fits     UGC     8  14.571953 -10.303926 -10.303926              False   

In [26]:
# === Cell 1: 路径与选项 ===
import os, shutil, pandas as pd

# 与之前保持一致
outdir      = r"E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3"   # FITS 输出目录
report_csv  = r"E:\NUS\galaxy\rc3\centered_check_report2.csv" # 评估报告
trash_dir   = os.path.join(outdir, "_no_coverage_quarantine") # 隔离文件夹

# True = 移动到隔离文件夹；False = 直接永久删除
SAFE_MOVE = True

os.makedirs(trash_dir, exist_ok=True)
print("[INFO] outdir =", outdir)
print("[INFO] report =", report_csv)
print("[INFO] trash_dir =", trash_dir, "(created)")
print("[INFO] mode =", "MOVE to quarantine" if SAFE_MOVE else "PERMANENT DELETE")


[INFO] outdir = E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3
[INFO] report = E:\NUS\galaxy\rc3\centered_check_report2.csv
[INFO] trash_dir = E:\NUS\galaxy\rc3\sdss_cutouts_hips2fits3\_no_coverage_quarantine (created)
[INFO] mode = MOVE to quarantine


In [27]:
# === Cell 2: 读取报告并筛选 ===
df = pd.read_csv(report_csv)
if "maybe_no_coverage" not in df.columns:
    raise RuntimeError("报告中没有列 `maybe_no_coverage`，请先运行评估步骤并生成该列。")

to_remove = df[df["maybe_no_coverage"] == True].copy()
print(f"[INFO] 标记为 'maybe_no_coverage==True' 的文件数：{len(to_remove)}")
to_remove[["file","altname","dr_arcsec"]].head(10)


[INFO] 标记为 'maybe_no_coverage==True' 的文件数：270


,file,altname,dr_arcsec
6,ESO__111-_12_103_g.fits,ESO 111- 12,3.045
7,ESO__149-_12_26_g.fits,ESO 149- 12,3.577
8,ESO__149-_13_57_g.fits,ESO 149- 13,4.013
9,ESO__149-_15_75_g.fits,ESO 149- 15,2.713
10,ESO__149-_16_106_g.fits,ESO 149- 16,3.115
11,ESO__193-_11_74_g.fits,ESO 193- 11,4.107
12,ESO__193-_14_87_g.fits,ESO 193- 14,2.155
13,ESO__193-_17_113_g.fits,ESO 193- 17,2.363
14,ESO__193-_18_122_g.fits,ESO 193- 18,3.660
15,ESO__193-_19_121_g.fits,ESO 193- 19,5.170


In [28]:
# === Cell 2: 读取报告并筛选（无语法错误版） ===
import os
import pandas as pd

def _try_read(path):
    """尝试多种方式读取CSV或TSV，返回(df, 参数)"""
    tried = []
    read_options = [
        {"sep": ",", "engine": "c"},
        {"sep": None, "engine": "python"},
        {"sep": "\t", "engine": "python"},
        {"sep": ",", "engine": "python", "encoding": "utf-8-sig"},
        {"sep": "\t", "engine": "python", "encoding": "utf-8-sig"},
    ]
    for kwargs in read_options:
        try:
            df = pd.read_csv(path, **kwargs)
            if df is not None and len(df.columns) > 0:
                return df, kwargs
            tried.append(kwargs)
        except Exception as e:
            tried.append({**kwargs, "err": str(e)})
    raise RuntimeError(f"无法解析文件：{path}\n尝试方式：{tried}")

# === 1) 检查文件存在与大小 ===
if not os.path.exists(report_csv):
    raise FileNotFoundError(f"找不到报告文件：{report_csv}")

if os.path.getsize(report_csv) == 0:
    raise RuntimeError(
        f"报告文件为空：{report_csv}\n"
        f"请先运行居中评估步骤（centered_check），生成 maybe_no_coverage 列。"
    )

# === 2) 读取 ===
df, used_kwargs = _try_read(report_csv)
print(f"[INFO] 读取成功，使用参数：{used_kwargs}, 形状={df.shape}")

# === 3) 清洗列名 ===
df.columns = [c.strip() for c in df.columns]

# 标准化 maybe_no_coverage 列
if "maybe_no_coverage" in df.columns:
    if df["maybe_no_coverage"].dtype == object:
        df["maybe_no_coverage"] = (
            df["maybe_no_coverage"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False, "1": True, "0": False})
            .fillna(False)
        )
else:
    # 兼容可能误写的列名
    alt_cols = [c for c in df.columns if c.replace(" ", "").lower() == "maybenocoverage"]
    if alt_cols:
        df.rename(columns={alt_cols[0]: "maybe_no_coverage"}, inplace=True)
    else:
        raise RuntimeError(
            "报告中没有列 `maybe_no_coverage`。\n"
            f"现有列：{list(df.columns)}"
        )

# === 4) 过滤 ===
to_remove = df[df["maybe_no_coverage"] == True].copy()
print(f"[INFO] 标记为 maybe_no_coverage==True 的文件数：{len(to_remove)}")

# === 5) 预览 ===
cols = [c for c in ["file", "altname", "dr_arcsec"] if c in to_remove.columns]
if cols:
    display(to_remove[cols].head(10))
else:
    print("[WARN] 预览列(file, altname, dr_arcsec)缺失，打印前几行：")
    print(to_remove.head(10))


[INFO] 读取成功，使用参数：{'sep': ',', 'engine': 'c'}, 形状=(750, 13)
[INFO] 标记为 maybe_no_coverage==True 的文件数：270


,file,altname,dr_arcsec
6,ESO__111-_12_103_g.fits,ESO 111- 12,3.045
7,ESO__149-_12_26_g.fits,ESO 149- 12,3.577
8,ESO__149-_13_57_g.fits,ESO 149- 13,4.013
9,ESO__149-_15_75_g.fits,ESO 149- 15,2.713
10,ESO__149-_16_106_g.fits,ESO 149- 16,3.115
11,ESO__193-_11_74_g.fits,ESO 193- 11,4.107
12,ESO__193-_14_87_g.fits,ESO 193- 14,2.155
13,ESO__193-_17_113_g.fits,ESO 193- 17,2.363
14,ESO__193-_18_122_g.fits,ESO 193- 18,3.660
15,ESO__193-_19_121_g.fits,ESO 193- 19,5.170


In [29]:
# === Cell 3: 移除 ===
removed, missing = [], []

for fname in to_remove["file"]:
    src = os.path.join(outdir, fname)
    if not os.path.isfile(src):
        missing.append(fname)
        continue

    if SAFE_MOVE:
        dst = os.path.join(trash_dir, fname)
        # 目标存在则先加后缀避免覆盖
        if os.path.exists(dst):
            base, ext = os.path.splitext(dst)
            k = 1
            while os.path.exists(dst):
                dst = f"{base}__dup{k}{ext}"
                k += 1
        shutil.move(src, dst)
    else:
        os.remove(src)
    removed.append(fname)

print(f"[DONE] 已{'移动' if SAFE_MOVE else '删除'} {len(removed)} 个文件。")
if missing:
    print(f"[WARN] 下列文件在目录中未找到（可能之前已处理）：")
    for x in missing[:10]:
        print(" -", x)


[DONE] 已移动 0 个文件。
[WARN] 下列文件在目录中未找到（可能之前已处理）：
 - ESO__111-_12_103_g.fits
 - ESO__149-_12_26_g.fits
 - ESO__149-_13_57_g.fits
 - ESO__149-_15_75_g.fits
 - ESO__149-_16_106_g.fits
 - ESO__193-_11_74_g.fits
 - ESO__193-_14_87_g.fits
 - ESO__193-_17_113_g.fits
 - ESO__193-_18_122_g.fits
 - ESO__193-_19_121_g.fits
